**Basic Line Environment without Q-Table**

In [13]:
import random
import numpy as np
import gymnasium as gym
from gymnasium import spaces

In [14]:
class LineWorldEnv(gym.Env):
    """A simple 1D environment for learning RL basics."""

    def __init__(self):
        super().__init__()

        # Actions: 0 = Left, 1 = Right
        self.action_space = spaces.Discrete(2)

        # Observations: positions 0, 1, 2, 3, 4
        self.observation_space = spaces.Discrete(5)

        self.goal = 4
        self.position = 0

    def reset(self, seed=None, options=None):
        """Reset the environment at the start of an episode."""
        super().reset(seed=seed)

        self.position = 0

        observation = self.position
        info = {}

        return observation, info

    def step(self, action):
        """Execute one action and return the resulting transition."""

        if not self.action_space.contains(action):
            raise ValueError(f"Invalid action: {action}")

        # Update the agent's position
        if action == 0:
            self.position = max(0, self.position - 1)
        elif action == 1:
            self.position = min(self.goal, self.position + 1)

        # Calculate reward and check termination
        terminated = self.position == self.goal

        if terminated:
            reward = 10
        else:
            reward = -1

        # No time limit in this initial environment
        truncated = False
        observation = self.position
        info = {}

        return observation, reward, terminated, truncated, info

    def render(self):
        """Print the current world."""
        world = ["-"] * (self.goal + 1)

        world[self.goal] = "G"
        world[self.position] = "A"

        print(" ".join(world))

In [15]:
env = LineWorldEnv()

In [16]:
# Run the environment

observation, info = env.reset(seed=42)
done = False
max_steps = 20

total_reward = 0

print("Starting LineWorld")
env.render()
print()

for step in range(max_steps):
# Random agent: it does not learn yet
    action = random.randint(0, 1) # here we can have our agent as well.

    observation, reward, terminated, truncated, info = (
        env.step(action)
    )

    total_reward += reward

    print(
        f"Step: {step + 1} | "
        f"Action: {'Left' if action == 0 else 'Right'} | "
        f"Position: {observation} | "
        f"Reward: {reward}"
    )

    env.render()
    print()

    if terminated or truncated:
        done = True
        break

print("Episode finished!" if done else "Step limit reached.")
print(f"Total reward: {total_reward}")

env.close()



Starting LineWorld
A - - - G

Step: 1 | Action: Right | Position: 1 | Reward: -1
- A - - G

Step: 2 | Action: Right | Position: 2 | Reward: -1
- - A - G

Step: 3 | Action: Right | Position: 3 | Reward: -1
- - - A G

Step: 4 | Action: Left | Position: 2 | Reward: -1
- - A - G

Step: 5 | Action: Left | Position: 1 | Reward: -1
- A - - G

Step: 6 | Action: Left | Position: 0 | Reward: -1
A - - - G

Step: 7 | Action: Right | Position: 1 | Reward: -1
- A - - G

Step: 8 | Action: Right | Position: 2 | Reward: -1
- - A - G

Step: 9 | Action: Right | Position: 3 | Reward: -1
- - - A G

Step: 10 | Action: Right | Position: 4 | Reward: 10
- - - - A

Episode finished!
Total reward: 1


**Basic Line Environment with Q-Table**

In [17]:
# start making the Q-table

# Q-table config

EPISODES = 1000
MAX_STEPS = 20

LEARNING_RATE = 0.1       # Alpha
DISCOUNT_FACTOR = 0.95    # Gamma

EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995

In [18]:
# Q-table helper functions

def choose_action(q_table, state, epsilon, env, rng):
    """Choose an action using epsilon-greedy selection."""

    # Explore: choose a random valid action
    if rng.random() < epsilon:
        return env.action_space.sample()

    # Exploit: choose the action with the highest Q-value
    return int(np.argmax(q_table[state]))


def train():
    env = LineWorldEnv()
    rng = np.random.default_rng(42)

    # Five states and two actions
    q_table = np.zeros(
        (env.observation_space.n, env.action_space.n)
    )

    epsilon = EPSILON_START
    episode_rewards = []

    # -------------------------
    # Training loop
    # -------------------------

    for episode in range(EPISODES):
        state, info = env.reset()
        total_reward = 0

        for step in range(MAX_STEPS):
            # 1. Choose action
            action = choose_action(
                q_table, state, epsilon, env, rng
            )

            # 2. Interact with environment
            next_state, reward, terminated, truncated, info = (
                env.step(action)
            )

            # 3. Calculate the Q-learning target
            old_value = q_table[state, action]

            if terminated or truncated:
                target = reward
            else:
                best_future_value = np.max(q_table[next_state])
                target = (
                    reward
                    + DISCOUNT_FACTOR * best_future_value
                )

            # 4. Update Q-table
            q_table[state, action] = old_value + (
                LEARNING_RATE * (target - old_value)
            )

            # 5. Move to next state
            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

        # Gradually reduce exploration
        epsilon = max(
            EPSILON_MIN,
            epsilon * EPSILON_DECAY
        )

        if (episode + 1) % 100 == 0:
            recent_rewards = episode_rewards[-100:]
            average_reward = np.mean(recent_rewards)

            print(
                f"Episode {episode + 1:4d} | "
                f"Avg reward: {average_reward:6.2f} | "
                f"Epsilon: {epsilon:.3f}"
            )

    env.close()

    return q_table, episode_rewards


def evaluate(q_table, episodes=5):
    """Test the learned policy without exploration."""

    env = LineWorldEnv()
    successful_episodes = 0

    print("\n========== EVALUATION ==========")

    for episode in range(episodes):
        state, info = env.reset()
        total_reward = 0

        print(f"\nEpisode {episode + 1}")
        env.render()

        for step in range(MAX_STEPS):
            # Pure exploitation: choose the best learned action
            action = int(np.argmax(q_table[state]))

            state, reward, terminated, truncated, info = (
                env.step(action)
            )

            total_reward += reward
            env.render()

            print(
                f"Action: {'Left' if action == 0 else 'Right'} | "
                f"Reward: {reward}"
            )

            if terminated or truncated:
                if terminated:
                    successful_episodes += 1
                break

        print(f"Total reward: {total_reward}")

    print(
        f"\nSuccess rate: "
        f"{successful_episodes}/{episodes}"
    )

    env.close()

In [19]:
# Run Episodes with Q-table in it.

q_table, episode_rewards = train()

print("\n========== LEARNED Q-TABLE ==========")
print("Columns: Left (0), Right (1)")
print(np.round(q_table, 2))

print("\n========== TRAINING SUMMARY ==========")
print(
    "Average reward over last 100 episodes:",
    round(float(np.mean(episode_rewards[-100:])), 2)
)

evaluate(q_table)

Episode  100 | Avg reward:  -0.94 | Epsilon: 0.606
Episode  200 | Avg reward:   3.95 | Epsilon: 0.367
Episode  300 | Avg reward:   5.67 | Epsilon: 0.222
Episode  400 | Avg reward:   6.30 | Epsilon: 0.135
Episode  500 | Avg reward:   6.53 | Epsilon: 0.082
Episode  600 | Avg reward:   6.69 | Epsilon: 0.049
Episode  700 | Avg reward:   6.90 | Epsilon: 0.030
Episode  800 | Avg reward:   6.98 | Epsilon: 0.018
Episode  900 | Avg reward:   6.96 | Epsilon: 0.011
Episode 1000 | Avg reward:   6.95 | Epsilon: 0.010

========== LEARNED Q-TABLE ==========
Columns: Left (0), Right (1)
[[ 4.44  5.72]
 [ 4.44  7.07]
 [ 5.72  8.5 ]
 [ 7.07 10.  ]
 [ 0.    0.  ]]

========== TRAINING SUMMARY ==========
Average reward over last 100 episodes: 6.95

========== EVALUATION ==========

Episode 1
A - - - G
- A - - G
Action: Right | Reward: -1
- - A - G
Action: Right | Reward: -1
- - - A G
Action: Right | Reward: -1
- - - - A
Action: Right | Reward: 10
Total reward: 7

Episode 2
A - - - G
- A - - G
Action: Righ